In [1]:
import requests, pandas as pd, re
from datetime import datetime, timedelta

import sys
import os

# Add the project root directory to the Python path
sys.path.append(os.path.abspath(".."))

# Import the API_KEY from your config module
from utils.config import NEWS_API_KEY

In [8]:
API_KEY = NEWS_API_KEY
QUERY = "FDA OR earnings OR acquisition OR upgrade OR contract"
LIMIT = 50
date_str = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
def get_general_news():
    url = "https://newsapi.org/v2/everything"
    params = {
        "q": QUERY,
        "from": date_str, # datetime.now().strftime('%Y-%m-%d'),
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": LIMIT,
        "apiKey": API_KEY
    }
    response = requests.get(url, params=params)
    # print("Status:", response.status_code)
    # print("Response:", response.json())
    if response.status_code != 200:
        print("Error fetching news:", response.status_code, response.text)
        return []
    else:
        return response.json().get("articles", [])

def extract_tickers(text):
    return re.findall(r'\b[A-Z]{2,5}\b', text)

def build_catalyst_news_list():
    articles = get_general_news()
    rows = []
    for a in articles:
        title, desc = a.get("title",""), a.get("description","")
        rows.append({
            "publishedAt": a.get("publishedAt"),
            "source": a.get("source",{}).get("name"),
            "title": title,
            "description": desc,
            "url": a.get("url")
        })
    df = pd.DataFrame(rows)
    out = f"../data/catalyst_news_{datetime.now().strftime('%Y-%m-%d')}.csv"
    df.to_csv(out, index=False)
    print("Saved:", out)

In [9]:
get_general_news()

[{'source': {'id': 'the-times-of-india', 'name': 'The Times of India'},
  'author': 'ET Online',
  'title': "Meghalya Murder Case: Sonam Raghuvanshi's secret plot of love & betrayal shocks the nation",
  'description': 'Raja Raghuvanshi murder case: A honeymoon in Meghalaya turned into a murder investigation after Raja Raghuvanshi was found dead in a forest with head injuries. His wife, Sonam Raghuvanshi, is the main suspect, accused of hiring contract killers with the help …',
  'url': 'https://economictimes.indiatimes.com/news/new-updates/meghalya-murder-case-sonam-raghuvanshis-secret-plot-of-love-betrayal-shocks-the-nation/articleshow/121759078.cms',
  'urlToImage': 'https://img.etimg.com/thumb/msid-121759469,width-1200,height-630,imgsize-220076,overlay-economictimes/articleshow.jpg',
  'publishedAt': '2025-06-10T17:35:41Z',
  'content': 'Raja Raghuvanshi murder case: What started as a dream honeymoon in the scenic hills of Meghalaya has turned into a gripping crime case involving m

In [10]:
build_catalyst_news_list()

Saved: ../data/catalyst_news_2025-06-11.csv


In [20]:
import requests
import json
import pandas as pd

# URLs for exchange symbol lists
url_nasdaq = "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/nasdaq/nasdaq_tickers.json"
url_nyse   = "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/nyse/nyse_tickers.json"
# url_amex   = "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/amex/amex_tickers.json"

def load_symbols(url):
    return json.loads(requests.get(url).text)

# Combine ticker symbols
tickers = set(load_symbols(url_nasdaq) + load_symbols(url_nyse)) # + load_symbols(url_amex))

In [22]:
df = pd.DataFrame({"symbol": sorted(tickers)})
# Save to CSV
df = pd.DataFrame({"symbol": sorted(tickers)})
df.to_csv("../data/tickers.csv", index=False)